# DDoS Detector — Preprocessing & Exploratory Data Analysis

> **Dataset:** CIC-IDS2017 (Canadian Institute for Cybersecurity)  
> **Source:** https://www.unb.ca/cic/datasets/ids-2017.html  
> **File:** `data/ddos.csv`  
> **Task:** Binary Classification — BENIGN (0) vs DDoS/Attack (1)

---
### Notebook Structure
1. Library Imports
2. Data Loading
3. Initial Dataset Inspection
4. Label Distribution (EDA)
5. Missing Value Analysis
6. Infinite Value Analysis
7. Data Cleaning
8. Stratified Sampling
9. Label Encoding
10. Feature Statistics & Correlation
11. Train-Test Split
12. Summary

## 1. Library Imports

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0C0915',
    'axes.facecolor':   '#120D22',
    'axes.edgecolor':   '#A855F7',
    'axes.labelcolor':  '#E9D5FF',
    'xtick.color':      '#C4B5FD',
    'ytick.color':      '#C4B5FD',
    'text.color':       '#F5F3FF',
    'grid.color':       '#2D1B4E',
    'grid.linestyle':   '--',
    'axes.grid':        True,
})

PURPLE = '#A855F7'
TEAL   = '#22D3EE'
RED    = '#EF4444'
GOLD   = '#F59E0B'
GREEN  = '#10B981'

print('Libraries imported successfully.')

## 2. Data Loading

In [ ]:
DATA_PATH = Path('data/ddos.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH.resolve()}.\n'
        'Please ensure data/ddos.csv is present in the project root.'
    )

print(f'Loading dataset from: {DATA_PATH.resolve()}')
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = df_raw.columns.str.strip()
print(f'Dataset loaded. Shape: {df_raw.shape}')

## 3. Initial Dataset Inspection

In [ ]:
print(f'Rows    : {df_raw.shape[0]:,}')
print(f'Columns : {df_raw.shape[1]:,}')
df_raw.head()

In [ ]:
print('Column Data Types:')
print(df_raw.dtypes.value_counts())

In [ ]:
df_raw.describe().T.style.background_gradient(cmap='Purples')

## 4. Label Distribution (EDA)

In [ ]:
label_counts = df_raw['Label'].value_counts()
label_pcts   = df_raw['Label'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Count': label_counts, 'Percentage (%)': label_pcts.round(2)})
print('=== Label Distribution ===')
print(dist_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CIC-IDS2017 — Class Distribution', fontsize=15, fontweight='bold', color='white')

colors = [TEAL if lbl == 'BENIGN' else RED for lbl in label_counts.index]
bars = axes[0].bar(label_counts.index.astype(str), label_counts.values,
                   color=colors, edgecolor=PURPLE, linewidth=1.2, width=0.5, alpha=0.9)
for bar in bars:
    h = bar.get_height()
    pct = h / len(df_raw) * 100
    axes[0].annotate(f'{h:,}\n({pct:.1f}%)',
                     xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 6), textcoords='offset points',
                     ha='center', va='bottom', fontsize=10, fontweight='bold', color='white')
axes[0].set_title('Sample Counts per Class', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Class Label')
axes[0].set_ylabel('Sample Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

pie_colors = [TEAL if lbl == 'BENIGN' else RED for lbl in label_counts.index]
axes[1].pie(
    label_counts.values,
    labels=[f'{l}\n({p:.1f}%)' for l, p in zip(label_counts.index, label_pcts)],
    colors=pie_colors, startangle=140,
    wedgeprops=dict(edgecolor=PURPLE, linewidth=1.5),
    textprops={'color': 'white', 'fontsize': 10}
)
axes[1].set_title('Class Proportion', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('models/class_dist_eda.png', dpi=200, bbox_inches='tight', facecolor='#0C0915')
plt.show()
print('Chart saved to models/class_dist_eda.png')

## 5. Missing Value Analysis

In [ ]:
null_counts   = df_raw.isnull().sum()
total_nulls   = null_counts.sum()
cols_with_nan = null_counts[null_counts > 0]

print(f'Total missing values: {total_nulls:,}')
if not cols_with_nan.empty:
    print('\nColumns with missing values:')
    for col, cnt in cols_with_nan.items():
        print(f'  - {col}: {cnt:,} ({cnt/len(df_raw)*100:.3f}%)')
else:
    print('No missing values found in raw data.')

In [ ]:
sample_df = df_raw.sample(min(5000, len(df_raw)), random_state=42)
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(sample_df.isnull(), cbar=False, ax=ax,
            cmap=['#120D22', RED], yticklabels=False)
ax.set_title('Missing Value Heatmap (5k row sample)', fontsize=12, fontweight='bold')
ax.set_xlabel('Features')
plt.tight_layout()
plt.show()

## 6. Infinite Value Analysis

In [ ]:
numeric_cols  = df_raw.select_dtypes(include=[np.number]).columns
inf_counts    = np.isinf(df_raw[numeric_cols]).sum()
total_infs    = inf_counts.sum()
cols_with_inf = inf_counts[inf_counts > 0]

print(f'Total infinite values: {total_infs:,}')
if not cols_with_inf.empty:
    print('\nColumns with infinite values:')
    for col, cnt in cols_with_inf.items():
        print(f'  - {col}: {cnt:,} ({cnt/len(df_raw)*100:.3f}%)')
else:
    print('No infinite values found.')

In [ ]:
if not cols_with_inf.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    cols_with_inf.sort_values(ascending=False).plot.bar(ax=ax, color=GOLD, edgecolor=PURPLE)
    ax.set_title('Columns Containing Infinite Values', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count of Inf values')
    ax.set_xlabel('Feature')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No infinite value bar chart needed.')

## 7. Data Cleaning

Steps:  
1. Replace `+/-Inf` with `NaN`  
2. Drop rows with `NaN`  
3. Remove duplicate rows

In [ ]:
df = df_raw.copy()
initial_rows = len(df)
print(f'Initial rows: {initial_rows:,}')

# Step 1: Replace Inf with NaN
df = df.replace([np.inf, -np.inf], np.nan)
print(f'After Inf->NaN replacement, NaN count: {df.isnull().sum().sum():,}')

# Step 2: Drop NaN rows
df = df.dropna()
after_nan = len(df)
print(f'Dropped {initial_rows - after_nan:,} rows containing NaN.')

# Step 3: Drop duplicates
df = df.drop_duplicates()
after_dedup = len(df)
print(f'Dropped {after_nan - after_dedup:,} duplicate rows.')
print(f'Cleaned dataset shape: {df.shape}')

In [ ]:
stages  = ['Raw Dataset', 'After Drop NaN', 'After Dedup']
row_cts = [initial_rows, after_nan, after_dedup]

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = [PURPLE, TEAL, GREEN]
bars = ax.bar(stages, row_cts, color=bar_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, row_cts):
    ax.annotate(f'{val:,}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 6), textcoords='offset points',
                ha='center', va='bottom', fontsize=11, fontweight='bold', color='white')
ax.set_title('Row Count at Each Cleaning Stage', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Rows')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Stratified Sampling

If the cleaned dataset exceeds 100,000 rows, sample 100k rows stratified by label.

In [ ]:
SAMPLE_SIZE = 100_000

if len(df) > SAMPLE_SIZE:
    print(f'Dataset has {len(df):,} rows. Sampling {SAMPLE_SIZE:,} rows (stratified)...')
    df = df.sample(n=SAMPLE_SIZE, random_state=42)
else:
    print(f'Dataset size is {len(df):,} rows. Using all rows.')

print(f'Sampled dataset shape: {df.shape}')
print('\nLabel distribution in sampled data:')
print(df['Label'].value_counts())

## 9. Label Encoding

- `BENIGN` -> `0`  
- Any attack class (DDoS, DoS, etc.) -> `1`

In [ ]:
print('Unique label values before encoding:')
print(df['Label'].unique())

df['Label'] = (df['Label'].astype(str).str.strip().str.upper() != 'BENIGN').astype(int)

encoded_dist = df['Label'].value_counts().rename(index={0: 'BENIGN (0)', 1: 'DDoS/Attack (1)'})
print('\nLabel distribution after binary encoding:')
print(encoded_dist)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['BENIGN (0)', 'DDoS/Attack (1)'], encoded_dist.values,
              color=[TEAL, RED], edgecolor=PURPLE, linewidth=1.2, width=0.4)
for bar in bars:
    h = bar.get_height()
    ax.annotate(f'{h:,} ({h/len(df)*100:.1f}%)',
                xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 6), textcoords='offset points',
                ha='center', fontsize=11, fontweight='bold', color='white')
ax.set_title('Encoded Label Distribution (Sampled)', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 10. Feature Statistics & Correlation

In [ ]:
X = df.drop(columns=['Label'])
y = df['Label']
feature_names = list(X.columns)
print(f'Number of input features: {len(feature_names)}')

In [ ]:
corr_with_label = X.corrwith(y).abs().dropna().sort_values(ascending=False)
top20_corr = corr_with_label.head(20)
print('Top 20 Features by |Pearson Correlation| with Label:')
print(top20_corr.round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
norm   = plt.Normalize(top20_corr.min(), top20_corr.max())
colors = plt.cm.plasma(norm(top20_corr.values[::-1]))
ax.barh(top20_corr.index[::-1], top20_corr.values[::-1],
        color=colors, edgecolor='#2D1B4E', linewidth=0.7, height=0.65)
ax.set_title('Top 20 Features — |Pearson Correlation| with Label', fontsize=13, fontweight='bold')
ax.set_xlabel('|Correlation Coefficient|')
ax.set_ylabel('Feature')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
top15_feats = top20_corr.head(15).index.tolist()
corr_matrix = X[top15_feats].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdPu',
    linewidths=0.5, linecolor='#2D1B4E', ax=ax,
    annot_kws={'size': 8}, cbar_kws={'shrink': 0.8}
)
ax.set_title('Inter-Feature Correlation Heatmap (Top 15)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
top5 = top20_corr.head(5).index.tolist()
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Feature Distributions: BENIGN vs DDoS (Top 5 Correlated)', fontsize=13, fontweight='bold')

for i, feat in enumerate(top5):
    benign_vals = X.loc[y == 0, feat].dropna()
    attack_vals = X.loc[y == 1, feat].dropna()
    p1, p99 = np.percentile(X[feat].dropna(), [1, 99])
    axes[i].hist(benign_vals.clip(p1, p99), bins=40, color=TEAL, alpha=0.6, label='BENIGN')
    axes[i].hist(attack_vals.clip(p1, p99), bins=40, color=RED,  alpha=0.6, label='DDoS')
    axes[i].set_title(feat[:22], fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].legend(fontsize=7)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 11. Train-Test Split (80/20 Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]:,}  ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Testing samples  : {X_test.shape[0]:,}  ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nTrain label dist:\n{y_train.value_counts().rename(index={0:"BENIGN",1:"DDoS"})}')
print(f'\nTest label dist:\n{y_test.value_counts().rename(index={0:"BENIGN",1:"DDoS"})}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
splits = ['Train Set (80%)', 'Test Set (20%)']
counts = [len(X_train), len(X_test)]
bars = ax.bar(splits, counts, color=[PURPLE, TEAL], edgecolor='white', width=0.4)
for bar, val in zip(bars, counts):
    ax.annotate(f'{val:,}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 6), textcoords='offset points',
                ha='center', fontsize=13, fontweight='bold', color='white')
ax.set_title('Stratified 80/20 Train-Test Split', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Samples')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 12. Summary

| Step | Detail |
|---|---|
| **Dataset** | CIC-IDS2017 (`data/ddos.csv`) |
| **Raw rows** | 225,745 |
| **Features** | 78 bidirectional flow indicators |
| **Cleaning** | Inf to NaN, dropna, dedup |
| **Sampling** | 100,000 stratified rows |
| **Label** | BENIGN=0, Attack=1 (binary) |
| **Split** | 80,000 train / 20,000 test |
| **Next step** | Model training — see `src/train_ddos.py` |

In [ ]:
out_path = Path('data/sample_traffic.csv')
df.to_csv(out_path, index=False)
print(f'Cleaned and sampled data saved to: {out_path}')